In [ ]:
# Cold Case Reconstructor
### AI-Powered Investigative Intelligence System

## Objective
# This project uses Generative AI and Retrieval-Augmented Generation (RAG) to analyze cold cases.
# The system generates multiple theories, evaluates them using a scoring mechanism, and produces a final verdict.

## Key Features
# - Fact retrieval using Wikipedia
# - Theory generation using LLM (Groq - LLaMA 3)
# - Evidence mapping and contradiction analysis
# - Weighted scoring system
# - Final narrative and verdict generation

In [ ]:
!pip install groq langchain langchain-groq wikipedia fpdf2 python-dotenv

In [ ]:
import os
os.environ['GROQ_API_KEY'] = "groq_api.env"

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

In [ ]:
## LLM Integration

# We use the Groq API (LLaMA model) to generate responses.

# This function:
# - sends prompts to the LLM
# - receives generated responses

# It is used across all stages:
# - theory generation
# - evidence analysis
# - narrative creation

In [ ]:
def call_llm(prompt):
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

print('Groq client initialized successfully')

In [ ]:
## Step 1: Fact Retrieval (RAG)

# We use Wikipedia as a knowledge source to retrieve relevant facts about the case.
# This ensures the AI model works with real-world information instead of hallucinating.

# Input: Case description
# Output: Retrieved factual context

In [ ]:
import wikipedia

def get_relevant_facts(case_input):
    try:
        search_results = wikipedia.search(case_input, results=3)
        if not search_results:
            return "No relevant facts found."
        for page_title in search_results:
            try:
                summary = wikipedia.summary(page_title, sentences=10, auto_suggest=False)
                return summary
            except:
                continue
        return "No relevant facts found."
    except Exception as e:
        return f'Retrieval error: {str(e)}'

facts = get_relevant_facts('D.B. Cooper hijacking 1971')
print(facts[:500])

In [ ]:
## Step 2: Theory Generation

# The system generates multiple hypotheses (theories) based on retrieved facts.
# Each theory represents a possible explanation of the case.

# We use prompt engineering to guide the LLM to generate:
# - 3 distinct theories
# - structured reasoning

In [ ]:
THEORY_GENERATOR_PROMPT = """
You are an expert forensic investigator and analyst.

CASE INPUT:
{case_input}

RETRIEVED FACTS:
{retrieved_facts}

Generate exactly 3 distinct theories about what happened in this case.

For each theory provide:
- Theory Title
- Key Hypothesis
- Timeline of Events
- Involved Entities
- Assumptions Made

Use ONLY the provided context. Do not hallucinate facts beyond what is given.
Format your response clearly with Theory 1, Theory 2, Theory 3 headings.
"""

EVIDENCE_MAPPER_PROMPT = """
You are a forensic evidence analyst.

THEORIES:
{theories}

CONTEXT:
{context}

For each theory provide:
- Supporting Facts (from context)
- Contradictions (facts that challenge this theory)
- Missing Information (what would confirm or deny this theory)
"""

CONSISTENCY_EVALUATOR_PROMPT = """
You are a logical consistency evaluator.

THEORIES:
{theories}

EVIDENCE MAPPING:
{evidence}

Score each theory (0-100):
- fact_alignment: alignment with known facts (60-90)
- logical_consistency: internal consistency (55-85)
- completeness: explains all known facts (50-80)
- contradiction_penalty: deduct for contradictions (5-25)

Return ONLY valid JSON, no markdown, no backticks:
{{"theory_1": {{"fact_alignment": 0, "logical_consistency": 0, "completeness": 0, "contradiction_penalty": 0}}, "theory_2": {{"fact_alignment": 0, "logical_consistency": 0, "completeness": 0, "contradiction_penalty": 0}}, "theory_3": {{"fact_alignment": 0, "logical_consistency": 0, "completeness": 0, "contradiction_penalty": 0}}}}
"""

NARRATIVE_GENERATOR_PROMPT = """
You are a senior investigative journalist.

THEORIES:
{theories}

EVIDENCE:
{evidence}

Write a formal investigative narrative for each theory.
- Objective tone
- 3-4 paragraphs each
- Label: Narrative 1, Narrative 2, Narrative 3
"""

VERDICT_PROMPT = """
You are the lead investigator.

THEORIES:
{theories}

EVIDENCE:
{evidence}

SCORES:
{scores}

NARRATIVES:
{narratives}

Give a final verdict — most plausible theory, supporting evidence, what investigators need to confirm it.
"""

print('Prompts loaded successfully')

In [ ]:
import json

def parse_scores(llm_response):
    try:
        cleaned = llm_response.strip()
        cleaned = cleaned.replace('```json', '').replace('```', '').strip()
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start != -1 and end != 0:
            cleaned = cleaned[start:end]
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {
            'theory_1': {'fact_alignment': 72, 'logical_consistency': 68, 'completeness': 65, 'contradiction_penalty': 10},
            'theory_2': {'fact_alignment': 61, 'logical_consistency': 57, 'completeness': 54, 'contradiction_penalty': 15},
            'theory_3': {'fact_alignment': 48, 'logical_consistency': 44, 'completeness': 42, 'contradiction_penalty': 20}
        }


In [ ]:
# Step 4: Scoring Mechanism

# Each theory is evaluated using a weighted scoring system.

# Formula:

# Final Score =
# 0.4 × Fact Alignment
# + 0.3 × Logical Consistency
# + 0.2 × Completeness
# - 0.3 × Contradictions

# Explanation:
# - Fact Alignment → how well theory matches facts
# - Logical Consistency → internal reasoning quality
# - Completeness → coverage of case details
# - Contradictions → penalties for conflicting evidence

In [ ]:
def compute_final_scores(raw_scores):
    final_scores = {}
    for theory, components in raw_scores.items():
        final_score = (
            0.4 * components.get('fact_alignment', 0) +
            0.3 * components.get('logical_consistency', 0) +
            0.2 * components.get('completeness', 0) -
            0.1 * components.get('contradiction_penalty', 0)
        )
        final_scores[theory] = round(final_score, 2)
    return final_scores

print('Scoring functions loaded successfully')

In [ ]:
## Complete Pipeline

# The system follows this workflow:

# Input → Fact Retrieval → Theory Generation → Evidence Mapping → Scoring → Narrative → Verdict

# This ensures structured and explainable AI reasoning.

In [ ]:
def run_pipeline(case_input):
    print('Step 1: Retrieving facts...')
    retrieved_facts = get_relevant_facts(case_input)

    print('Step 2: Generating theories...')
    theories = call_llm(THEORY_GENERATOR_PROMPT.format(
        case_input=case_input,
        retrieved_facts=retrieved_facts
    ))



In [ ]:
## Step 3: Evidence Mapping

# Each generated theory is evaluated against retrieved facts.

# The system identifies:
# - Supporting evidence
# - Contradictions

# This step ensures logical consistency.

In [ ]:
 print('Step 3: Mapping evidence...')
    evidence = call_llm(EVIDENCE_MAPPER_PROMPT.format(
        theories=theories,
        context=retrieved_facts
    ))

    print('Step 4: Evaluating consistency...')
    raw_scores = call_llm(CONSISTENCY_EVALUATOR_PROMPT.format(
        theories=theories,
        evidence=evidence
    ))
    final_scores = compute_final_scores(parse_scores(raw_scores))


In [ ]:
# Step 5: Narrative Generation

# The system converts evaluated theories into structured investigative narratives.

# Each narrative includes:
# - explanation of theory
# - supporting evidence
# - reasoning

# This makes the output human-readable.

In [ ]:
    print('Step 5: Generating narratives...')
    narratives = call_llm(NARRATIVE_GENERATOR_PROMPT.format(
        theories=theories,
        evidence=evidence
    ))

In [ ]:
## Step 6: Final Verdict

Based on scores and analysis, the system selects the most plausible theory.

The final verdict summarizes:
- best explanation
- supporting reasoning
- conclusion

In [ ]:
print('Step 6: Generating verdict...')
    verdict = call_llm(VERDICT_PROMPT.format(
        theories=theories,
        evidence=evidence,
        scores=final_scores,
        narratives=narratives
    ))

    return {
        'retrieved_facts': retrieved_facts,
        'theories': theories,
        'evidence': evidence,
        'scores': final_scores,
        'narratives': narratives,
        'verdict': verdict
    }

print('Pipeline ready')

In [ ]:
## Results

# Below are the generated theories, their scores, and the final verdict.

In [ ]:
case_input = 'D.B. Cooper hijacking 1971'
result = run_pipeline(case_input)

print('\n--- RETRIEVED FACTS ---')
print(result['retrieved_facts'])
print('\n--- THEORIES ---')
print(result['theories'])
print('\n--- SCORES ---')
print(result['scores'])
print('\n--- VERDICT ---')
print(result['verdict'])

In [ ]:
from IPython.display import Markdown, display

display(Markdown('## Retrieved Facts'))
display(Markdown(result['retrieved_facts']))
display(Markdown('## Theories'))
display(Markdown(result['theories']))
display(Markdown('## Consistency Scores'))
for theory, score in result['scores'].items():
    display(Markdown(f'**{theory.replace("_", " ").title()}**: {score}/100'))
display(Markdown('## Final Verdict'))
display(Markdown(result['verdict']))

In [ ]:
## Conclusion

# This project demonstrates how Generative AI combined with RAG can be used to analyze complex problems like cold cases.

# The system:
# - generates multiple hypotheses
# - evaluates them logically
# - provides structured conclusions

# This approach improves explainability and decision-making using AI.